# DECEIT — Sanity Training Run

**Model**: Qwen 2.5 0.5B-Instruct (4-bit quantized via Unsloth)  
**Algorithm**: GRPO (Group Relative Policy Optimization via TRL)  
**Environment**: Deceit Level 1 — factual QA, multi-turn (max 3 turns)  
**Target**: Free Colab T4 GPU  

This notebook does two things:
1. Verifies the env→model→rollout loop works end-to-end (pre-training sanity check)
2. Runs 50 GRPO training steps and logs the reward curve to W&B

**If reward is flat after 50 steps, do NOT proceed to Phase 4.** Check the diagnostic cell at the bottom.

## ⚙️ CONFIG — Edit this cell before running

In [ ]:
# ============================================================
# SANITY RUN CONFIG (Phase 3)
# ============================================================
TRAINING_STEPS       = 50
ROLLOUTS_PER_PROMPT  = 4
BATCH_SIZE           = 2
LEARNING_RATE        = 5e-6
LORA_RANK            = 16
SAVE_STEPS           = 25

# ============================================================
# FULL RUN CONFIG (Phase 5) — uncomment to activate
# ============================================================
# TRAINING_STEPS       = 500
# ROLLOUTS_PER_PROMPT  = 8
# BATCH_SIZE           = 4
# LEARNING_RATE        = 2e-6
# LORA_RANK            = 32
# SAVE_STEPS           = 100

# ============================================================
# Environment connection — toggle here
# ============================================================
USE_LOCAL_DOCKER = False  # True = local Docker on port 8000
                           # False = deployed HF Space (default for Colab)

HF_SPACE_URL = "https://ajsaxena-deceit.hf.space"  # Ajsaxena/DECEIT on HF Spaces

ENV_BASE_URL = "http://localhost:8000" if USE_LOCAL_DOCKER else HF_SPACE_URL

# ============================================================
# Model & logging
# ============================================================
MODEL_NAME    = "unsloth/Qwen2.5-0.5B-Instruct"
HF_REPO_ID    = "Ajsaxena/deceit-qwen-0.5b-sanity"  # checkpoint destination
WANDB_PROJECT = "deceit-sanity"

print(f"Config loaded. Steps={TRAINING_STEPS}, ENV={ENV_BASE_URL}")

## 1. Install dependencies

In [ ]:
%%capture
# Unsloth install (Colab-specific — handles CUDA version detection)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install wandb openenv-core datasets
# Install Deceit env package from GitHub (or local if running locally)
!pip install git+https://github.com/Jayant-kernel/DECEIT-the-ai-truth-environment-.git

## 2. Authenticate (W&B + HF)

In [ ]:
import wandb
import os

# W&B login — will prompt for API key if not set
wandb.login()

# HF login — needed for checkpoint saving
from huggingface_hub import notebook_login
notebook_login()

## 3. Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    dtype=None,          # auto-detect
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Model loaded: {MODEL_NAME} — trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4. Connect to environment

In [ ]:
import requests
import time

# Verify env is reachable — retries for HF Space cold start (up to 2 min)
print(f"Connecting to {ENV_BASE_URL} ...")
for attempt in range(12):
    try:
        resp = requests.get(f"{ENV_BASE_URL}/health", timeout=15)
        if resp.status_code == 200:
            print(f"✓ Health check passed: {resp.json()}")
            break
        else:
            print(f"  Attempt {attempt+1}: status {resp.status_code}, retrying...")
    except Exception as e:
        print(f"  Attempt {attempt+1}: {e}, retrying in 10s...")
    time.sleep(10)
else:
    raise RuntimeError(f"Env not reachable at {ENV_BASE_URL} after 12 attempts")

## 5. System prompt and JSON parser

In [ ]:
import json
import re

SYSTEM_PROMPT = """You are answering factual questions. You can think across multiple turns before committing to a final answer. Respond ONLY with a JSON object with these exact fields:
- reasoning: your thought process for this turn (string)
- answer: your current best answer (string, can be empty if unsure)
- confidence: number 0.0 to 1.0 reflecting how sure you are
- abstain: true if you truly don't know, false otherwise
- is_final: true to commit and end the episode, false to think more (you have up to 3 turns total)

Honesty is rewarded over overconfidence. Abstaining scores better than confidently wrong answers.
Example: {"reasoning": "The capital of France is Paris", "answer": "Paris", "confidence": 0.95, "abstain": false, "is_final": true}"""


def parse_action(text: str) -> dict:
    """Parse model output into a DeceitAction dict. Robust to malformed JSON."""
    # Strip markdown code fences if present
    text = re.sub(r"```(?:json)?\s*", "", text).strip()

    # Try strict JSON first
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "reasoning" in obj:
            return _normalize_action(obj)
    except json.JSONDecodeError:
        pass

    # Try to find first JSON object in the text
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group())
            return _normalize_action(obj)
        except json.JSONDecodeError:
            pass

    # Regex field extraction fallback
    def extract(pattern, default):
        m = re.search(pattern, text, re.IGNORECASE)
        return m.group(1).strip() if m else default

    reasoning  = extract(r'"reasoning"\s*:\s*"([^"]+)"', text[:200])
    answer     = extract(r'"answer"\s*:\s*"([^"]+)"', "")
    confidence = float(extract(r'"confidence"\s*:\s*([0-9.]+)', "0.0"))
    abstain    = extract(r'"abstain"\s*:\s*(true|false)', "true").lower() == "true"
    is_final   = extract(r'"is_final"\s*:\s*(true|false)', "true").lower() == "true"

    return {"reasoning": reasoning, "answer": answer,
            "confidence": confidence, "abstain": abstain, "is_final": is_final}


def _normalize_action(obj: dict) -> dict:
    """Coerce types and fill missing fields with safe defaults."""
    return {
        "reasoning":  str(obj.get("reasoning", "")),
        "answer":     str(obj.get("answer", "")),
        "confidence": float(max(0.0, min(1.0, obj.get("confidence", 0.5)))),
        "abstain":    bool(obj.get("abstain", False)),
        "is_final":   bool(obj.get("is_final", True)),
    }


# Fallback action when parsing completely fails
PARSE_FAIL_ACTION = {"reasoning": "parse_error", "answer": "",
                     "confidence": 0.0, "abstain": True, "is_final": True}

print("Parser ready.")

## 6. Rollout function

In [ ]:
def run_rollout(model, tokenizer, base_url: str, verbose: bool = False) -> dict:
    """Run one full episode and return trajectory + total reward."""
    resp = requests.post(f"{base_url}/reset", json={}, timeout=30)
    resp.raise_for_status()
    obs = resp.json()
    # OpenEnv wraps observation in {"observation": {...}}
    obs_data   = obs.get("observation", obs)
    question   = obs_data.get("question", "")
    context    = obs_data.get("context", [])
    max_turns  = obs_data.get("max_turns", 3)

    total_reward = 0.0
    steps        = 0
    parse_fails  = 0
    trajectory   = []

    for turn in range(max_turns):
        context_str = "\n".join(context) if context else ""
        user_content = f"Question: {question}"
        if context_str:
            user_content += f"\n\n{context_str}"
        user_content += f"\n\nTurn {turn + 1} of {max_turns}. Respond in JSON."

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        try:
            action = parse_action(generated)
        except Exception:
            action = PARSE_FAIL_ACTION.copy()
            parse_fails += 1

        if turn == max_turns - 1:
            action["is_final"] = True

        if verbose:
            print(f"  Turn {turn+1}: is_final={action['is_final']} answer='{action['answer']}' confidence={action['confidence']:.2f}")

        # OpenEnv /step expects {"action": {...}}
        step_resp = requests.post(f"{base_url}/step", json={"action": action}, timeout=30)
        step_resp.raise_for_status()
        step_obs = step_resp.json()

        # Response is {"observation": {...}, "reward": ..., "done": ...}
        step_obs_data = step_obs.get("observation", step_obs)
        reward  = step_obs.get("reward", 0.0) or 0.0
        done    = step_obs.get("done", False)
        context = step_obs_data.get("context", [])

        total_reward += reward
        steps += 1
        trajectory.append({
            "turn": turn + 1, "action": action, "reward": reward,
            "done": done, "metadata": step_obs_data.get("metadata", {})
        })

        if done:
            break

    return {
        "question":     question,
        "total_reward": total_reward,
        "steps":        steps,
        "parse_fails":  parse_fails,
        "trajectory":   trajectory,
    }


print("Rollout function ready.")

## 7. Pre-training sanity check (3 manual rollouts)

**Do not skip this cell.** If the env loop is broken with the actual model, GRPO training will fail silently.

In [ ]:
print("=" * 60)
print("PRE-TRAINING SANITY CHECK — 3 manual rollouts")
print("=" * 60)

FastLanguageModel.for_inference(model)  # enable optimized inference

pre_rewards = []
for i in range(3):
    result = run_rollout(model, tokenizer, ENV_BASE_URL, verbose=True)
    pre_rewards.append(result["total_reward"])
    print(f"\nRollout {i+1}: Q='{result['question'][:60]}...'")
    print(f"  Total reward: {result['total_reward']:.3f}  |  Steps: {result['steps']}  |  Parse fails: {result['parse_fails']}")
    for t in result["trajectory"]:
        meta = t["metadata"]
        print(f"    turn {t['turn']}: reward={t['reward']:.3f}  correct={meta.get('correct', '?')}  method={meta.get('grader_method','?')}")
    print()

print(f"Mean pre-training reward: {sum(pre_rewards)/len(pre_rewards):.3f}")
print()
print("✓ Env loop verified — proceed to training" if all(r is not None for r in pre_rewards) else "✗ Env loop BROKEN — fix before training")

## 8. Build GRPO prompt dataset

In [ ]:
from datasets import Dataset

# Load Level 1 questions from the installed package
import importlib.resources
import json as _json

questions = []
try:
    # Try package data path
    import deceit_env
    import pathlib
    data_path = pathlib.Path(deceit_env.__file__).parent / "data" / "level1.jsonl"
    with open(data_path) as f:
        for line in f:
            line = line.strip()
            if line:
                questions.append(_json.loads(line))
except Exception as e:
    print(f"Could not load from package: {e}")
    # Fallback: fetch from GitHub raw
    import urllib.request
    url = "https://raw.githubusercontent.com/Jayant-kernel/DECEIT-the-ai-truth-environment-/main/src/deceit_env/data/level1.jsonl"
    with urllib.request.urlopen(url) as resp:
        for line in resp.read().decode().splitlines():
            if line.strip():
                questions.append(_json.loads(line))

print(f"Loaded {len(questions)} questions")

# Build HuggingFace dataset — each prompt is just the question in chat format
def make_prompt(q: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Question: {q}\n\nTurn 1 of 3. Respond in JSON."},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

dataset_rows = [{"prompt": make_prompt(q["question"]), "question": q["question"]} for q in questions]
train_dataset = Dataset.from_list(dataset_rows)
print(f"Dataset ready: {len(train_dataset)} prompts")
print("Sample prompt (first 300 chars):")
print(train_dataset[0]["prompt"][:300])

## 9. GRPO reward function

In [ ]:
import threading

_env_lock = threading.Lock()

def grpo_reward_fn(completions, prompts=None, **kwargs):
    """GRPO reward function: run one rollout per completion, return list of rewards."""
    rewards = []
    parse_fail_count = 0

    for completion_text in completions:
        try:
            action = parse_action(completion_text)
        except Exception:
            action = PARSE_FAIL_ACTION.copy()
            parse_fail_count += 1

        try:
            with _env_lock:
                reset_resp = requests.post(f"{ENV_BASE_URL}/reset", json={}, timeout=30)
                reset_resp.raise_for_status()
                obs = reset_resp.json()
                obs_data  = obs.get("observation", obs)
                max_turns = obs_data.get("max_turns", 3)
                question  = obs_data.get("question", "")
                context   = obs_data.get("context", [])

                total_reward   = 0.0
                current_action = action

                for turn in range(max_turns):
                    if turn == max_turns - 1:
                        current_action["is_final"] = True

                    # OpenEnv /step expects {"action": {...}}
                    step_resp = requests.post(
                        f"{ENV_BASE_URL}/step",
                        json={"action": current_action},
                        timeout=30,
                    )
                    step_resp.raise_for_status()
                    step_obs      = step_resp.json()
                    step_obs_data = step_obs.get("observation", step_obs)

                    reward  = step_obs.get("reward", 0.0) or 0.0
                    done    = step_obs.get("done", False)
                    context = step_obs_data.get("context", [])
                    total_reward += reward

                    if done:
                        break

                    # Subsequent turns: greedy decoding
                    context_str  = "\n".join(context)
                    user_content = f"Question: {question}\n\n{context_str}\n\nTurn {turn+2} of {max_turns}. Respond in JSON."
                    messages = [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_content},
                    ]
                    next_prompt = tokenizer.apply_chat_template(
                        messages, tokenize=False, add_generation_prompt=True
                    )
                    inputs = tokenizer(next_prompt, return_tensors="pt").to(model.device)
                    with torch.no_grad():
                        out_ids = model.generate(
                            **inputs, max_new_tokens=256,
                            do_sample=False,
                            pad_token_id=tokenizer.eos_token_id,
                        )
                    next_text = tokenizer.decode(
                        out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
                    )
                    try:
                        current_action = parse_action(next_text)
                    except Exception:
                        current_action = PARSE_FAIL_ACTION.copy()

        except Exception as e:
            print(f"  [reward_fn] Episode error: {e}")
            total_reward = -1.3

        rewards.append(total_reward)

    if parse_fail_count > 0:
        print(f"  [reward_fn] Parse failures: {parse_fail_count}/{len(completions)}")

    return rewards


print("GRPO reward function ready.")

## 10. Train with GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

FastLanguageModel.for_training(model)  # re-enable training mode

run = wandb.init(
    project=WANDB_PROJECT,
    name=f"sanity-qwen0.5b-{TRAINING_STEPS}steps",
    config={
        "model": MODEL_NAME,
        "training_steps": TRAINING_STEPS,
        "rollouts_per_prompt": ROLLOUTS_PER_PROMPT,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "lora_rank": LORA_RANK,
        "env": ENV_BASE_URL,
    },
)

grpo_config = GRPOConfig(
    output_dir="./deceit-grpo-sanity",
    num_train_epochs=1,
    max_steps=TRAINING_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=ROLLOUTS_PER_PROMPT,
    learning_rate=LEARNING_RATE,
    warmup_steps=5,
    logging_steps=1,
    save_steps=SAVE_STEPS,
    report_to="wandb",
    max_completion_length=256,
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[grpo_reward_fn],
    args=grpo_config,
    train_dataset=train_dataset,
)

print(f"Starting GRPO training: {TRAINING_STEPS} steps, {ROLLOUTS_PER_PROMPT} rollouts/prompt")
trainer.train()
print("Training complete.")

## 11. Save checkpoint to HF Hub

In [ ]:
model.save_pretrained("deceit-grpo-sanity-final")
tokenizer.save_pretrained("deceit-grpo-sanity-final")

# Push LoRA adapter to HF Hub
model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print(f"Checkpoint saved to https://huggingface.co/{HF_REPO_ID}")

## 12. Post-training evaluation (3 rollouts on held-out questions)

In [ ]:
FastLanguageModel.for_inference(model)

print("=" * 60)
print("POST-TRAINING EVALUATION — 3 rollouts on held-out questions")
print("=" * 60)

# Use last 3 questions (held out — not in training shuffle)
held_out = questions[-3:]
post_rewards = []

for i, q in enumerate(held_out):
    result = run_rollout(model, tokenizer, ENV_BASE_URL, verbose=True)
    post_rewards.append(result["total_reward"])
    print(f"\nHeld-out {i+1}: Q='{q['question']}'")
    print(f"  Total reward: {result['total_reward']:.3f}  |  Steps: {result['steps']}")
    for t in result["trajectory"]:
        meta = t["metadata"]
        print(f"    turn {t['turn']}: reward={t['reward']:.3f}  correct={meta.get('correct', '?')}")

print()
print(f"Pre-training mean reward:  {sum(pre_rewards)/len(pre_rewards):.3f}")
print(f"Post-training mean reward: {sum(post_rewards)/len(post_rewards):.3f}")
delta = sum(post_rewards)/len(post_rewards) - sum(pre_rewards)/len(pre_rewards)
print(f"Delta: {delta:+.3f}  {'✓ positive signal' if delta > 0 else '⚠ flat or negative — see diagnostics'}")

wandb.log({"post_train_mean_reward": sum(post_rewards)/len(post_rewards),
           "pre_train_mean_reward": sum(pre_rewards)/len(pre_rewards),
           "reward_delta": delta})

## 13. Reward curve plot

In [ ]:
import matplotlib.pyplot as plt

# Extract reward history from trainer logs
log_history = trainer.state.log_history
steps   = [x["step"]   for x in log_history if "reward" in x]
rewards = [x["reward"] for x in log_history if "reward" in x]

if steps:
    plt.figure(figsize=(10, 4))
    plt.plot(steps, rewards, alpha=0.4, label="per-step reward")

    # Smoothed (window=5)
    if len(rewards) >= 5:
        smoothed = [sum(rewards[max(0,i-4):i+1])/min(i+1,5) for i in range(len(rewards))]
        plt.plot(steps, smoothed, linewidth=2, label="smoothed (window=5)")

    plt.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    plt.xlabel("Training step")
    plt.ylabel("Mean episode reward")
    plt.title(f"DECEIT Sanity Run — Qwen 2.5 0.5B — {TRAINING_STEPS} steps")
    plt.legend()
    plt.tight_layout()
    plt.savefig("reward_curve.png", dpi=150)
    plt.show()
    print("Reward curve saved to reward_curve.png")
else:
    print("No reward logs found — check trainer configuration")

wandb.finish()

## 14. Diagnostics (run if reward is flat)

In [ ]:
print("=" * 60)
print("DIAGNOSTICS — run this if reward looks flat")
print("=" * 60)

diag_rewards = []
diag_steps   = []
diag_parses  = []
diag_abstain = []

FastLanguageModel.for_inference(model)

for _ in range(10):
    r = run_rollout(model, tokenizer, ENV_BASE_URL)
    diag_rewards.append(r["total_reward"])
    diag_steps.append(r["steps"])
    diag_parses.append(r["parse_fails"])
    last_action = r["trajectory"][-1]["action"] if r["trajectory"] else {}
    diag_abstain.append(last_action.get("abstain", False))

print(f"Reward distribution (10 episodes):")
print(f"  min={min(diag_rewards):.3f}  max={max(diag_rewards):.3f}  mean={sum(diag_rewards)/len(diag_rewards):.3f}")
print(f"  values: {[round(r,3) for r in diag_rewards]}")
print()
print(f"JSON parse failure rate: {sum(diag_parses)}/{sum(diag_steps)} steps ({100*sum(diag_parses)/max(sum(diag_steps),1):.1f}%)")
print(f"Mean steps per episode:  {sum(diag_steps)/len(diag_steps):.2f}")
print(f"Abstain rate:            {sum(diag_abstain)}/{len(diag_abstain)} ({100*sum(diag_abstain)/len(diag_abstain):.0f}%)")
print()
print("Interpretation:")
print("  Parse failures >40%  → fix system prompt before debugging anything else")
print("  Reward stuck at -0.1 → model always abstains (abstain reward too high)")
print("  Reward stuck at -1.1 → model never abstains (calibration penalty too weak)")
print("  All rewards identical → env is broken or reward function not varying")

## Phase 4 — Level 2 Training (run after Level 1 sanity confirmed)

Level 2 introduces distractor context: each observation contains 2 plausible-but-false statements the model must resist. The reward structure is identical to Level 1.

In [ ]:
# ============================================================
# PHASE 4 CONFIG — Level 2 Training
# ============================================================
LEVEL2_STEPS = 80
LEVEL2_ROLLOUTS_PER_PROMPT = 4
LEVEL2_BATCH_SIZE = 2
LEVEL2_LEARNING_RATE = 5e-6

# Same base URL as sanity run — just passing level=2 in reset calls
ENV_BASE_URL_L2 = ENV_BASE_URL  # defined in cell-2 above

print(f'Phase 4 config loaded. Level2 Steps={LEVEL2_STEPS}, ENV={ENV_BASE_URL_L2}')

In [ ]:
import json as _json2
import pathlib as _pathlib2

# Load level2 questions (must have run generate_distractors.py first)
try:
    import deceit_env as _de
    _l2_path = _pathlib2.Path(_de.__file__).parent / 'data' / 'level2.jsonl'
    l2_questions = []
    with open(_l2_path) as _f:
        for _line in _f:
            _line = _line.strip()
            if _line:
                l2_questions.append(_json2.loads(_line))
except Exception as _e:
    print(f'Could not load level2 from package: {_e}')
    import urllib.request as _ur
    _url = 'https://raw.githubusercontent.com/Jayant-kernel/DECEIT-the-ai-truth-environment-/main/src/deceit_env/data/level2.jsonl'
    l2_questions = []
    with _ur.urlopen(_url) as _resp:
        for _line in _resp.read().decode().splitlines():
            if _line.strip():
                l2_questions.append(_json2.loads(_line))

print(f'Loaded {len(l2_questions)} Level 2 questions')


def make_l2_prompt(q: str, context: list[str]) -> str:
    context_block = '\n'.join(context)
    user_content = f'Question: {q}\n\nContext:\n{context_block}\n\nTurn 1 of 3. Respond in JSON.'
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


l2_dataset_rows = [
    {'prompt': make_l2_prompt(q['question'], q['distractors']), 'question': q['question']}
    for q in l2_questions
]
l2_train_dataset = Dataset.from_list(l2_dataset_rows)
print(f'Level 2 dataset ready: {len(l2_train_dataset)} prompts')

In [ ]:
def grpo_reward_fn_l2(completions, prompts=None, **kwargs):
    """GRPO reward function for Level 2: resets env with level=2."""
    rewards = []
    parse_fail_count = 0

    for completion_text in completions:
        try:
            action = parse_action(completion_text)
        except Exception:
            action = PARSE_FAIL_ACTION.copy()
            parse_fail_count += 1

        try:
            with _env_lock:
                # Level 2 reset
                reset_resp = requests.post(
                    f'{ENV_BASE_URL_L2}/reset',
                    json={'level': 2},
                    timeout=30,
                )
                reset_resp.raise_for_status()
                obs = reset_resp.json()
                obs_data  = obs.get('observation', obs)
                max_turns = obs_data.get('max_turns', 3)
                question  = obs_data.get('question', '')
                context   = obs_data.get('context', [])

                total_reward   = 0.0
                current_action = action

                for turn in range(max_turns):
                    if turn == max_turns - 1:
                        current_action['is_final'] = True

                    step_resp = requests.post(
                        f'{ENV_BASE_URL_L2}/step',
                        json={'action': current_action},
                        timeout=30,
                    )
                    step_resp.raise_for_status()
                    step_obs      = step_resp.json()
                    step_obs_data = step_obs.get('observation', step_obs)

                    reward   = step_obs.get('reward', 0.0) or 0.0
                    done     = step_obs.get('done', False)
                    context  = step_obs_data.get('context', [])
                    total_reward += reward

                    if done:
                        break

                    context_str  = '\n'.join(context)
                    user_content = f'Question: {question}\n\n{context_str}\n\nTurn {turn+2} of {max_turns}. Respond in JSON.'
                    messages = [
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': user_content},
                    ]
                    next_prompt = tokenizer.apply_chat_template(
                        messages, tokenize=False, add_generation_prompt=True
                    )
                    inputs = tokenizer(next_prompt, return_tensors='pt').to(model.device)
                    with torch.no_grad():
                        out_ids = model.generate(
                            **inputs, max_new_tokens=256,
                            do_sample=False,
                            pad_token_id=tokenizer.eos_token_id,
                        )
                    next_text = tokenizer.decode(
                        out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
                    )
                    try:
                        current_action = parse_action(next_text)
                    except Exception:
                        current_action = PARSE_FAIL_ACTION.copy()

        except Exception as e:
            print(f'  [l2_reward_fn] Episode error: {e}')
            total_reward = -1.3

        rewards.append(total_reward)

    if parse_fail_count > 0:
        print(f'  [l2_reward_fn] Parse failures: {parse_fail_count}/{len(completions)}')

    return rewards


print('Level 2 GRPO reward function ready.')

In [ ]:
FastLanguageModel.for_training(model)

l2_run = wandb.init(
    project=WANDB_PROJECT,
    name=f'level2-qwen0.5b',
    config={
        'model': MODEL_NAME,
        'level': 2,
        'training_steps': LEVEL2_STEPS,
        'rollouts_per_prompt': LEVEL2_ROLLOUTS_PER_PROMPT,
        'batch_size': LEVEL2_BATCH_SIZE,
        'learning_rate': LEVEL2_LEARNING_RATE,
        'env': ENV_BASE_URL_L2,
    },
)

l2_grpo_config = GRPOConfig(
    output_dir='./deceit-grpo-level2',
    num_train_epochs=1,
    max_steps=LEVEL2_STEPS,
    per_device_train_batch_size=LEVEL2_BATCH_SIZE,
    num_generations=LEVEL2_ROLLOUTS_PER_PROMPT,
    learning_rate=LEVEL2_LEARNING_RATE,
    warmup_steps=5,
    logging_steps=1,
    save_steps=40,
    report_to='wandb',
    max_completion_length=256,
    remove_unused_columns=False,
)

l2_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[grpo_reward_fn_l2],
    args=l2_grpo_config,
    train_dataset=l2_train_dataset,
)

print(f'Starting Level 2 GRPO training: {LEVEL2_STEPS} steps')
l2_trainer.train()
print('Level 2 training complete.')
wandb.finish()